# Build a mini-Leonida 🏙️
### The intelligence layer that makes infrastructure aware — on Google Cloud
*GDG DevFest Siliguri · Kazi Haque*

In GTA VI, the city of Leonida feels alive because everything in it is **aware** of everything else.
In this notebook you'll build a tiny version of that for a real campus, in three missions:

| Mission | Layer | What you build |
|---|---|---|
| 1 | **The map** | Simulated telemetry from a chiller, an air handler and a hall full of people |
| 2 | **Relationships** | A graph of who affects whom |
| 3 | **Awareness** | A Gemini agent that reasons across both, with safety guardrails (the *wanted-level ladder*) |

**Runs on free tools:** Google Colab + a free Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey).
No key? Every cell still runs — Mission 3 falls back to a sample answer so you can see the flow.

All data here is synthetic.

In [ ]:
!pip -q install -U google-genai networkx pandas matplotlib

## Mission 1 · Build the map 🗺️
Two hours of a DevFest morning, every 5 minutes. At **10:40** the chiller's condenser pump quietly starts losing flow.
Notice that both machines keep reporting `RUNNING` the whole time.

In [ ]:
import pandas as pd, numpy as np

def simulate(drift_starts_min=40, drift_rate=0.48, crowd_growth=4.5):
    times = pd.date_range("2026-11-14 10:00", "2026-11-14 12:00", freq="5min")
    rows = []
    for t in times:
        m = (t - times[0]).seconds / 60
        flow = 100 if m < drift_starts_min else max(55, 100 - (m - drift_starts_min) * drift_rate)
        chw = 7.0 + (100 - flow) * 0.12                 # chilled water warms as pump flow drops
        people = int(min(620, 250 + m * crowd_growth))   # the hall fills up
        outdoor = 29 + m * 0.025                         # it gets hotter outside
        ahu = 13 + (chw - 7) * 1.15                      # air handler can't cool the air as well
        hall = 23.6 + (ahu - 13) * 0.55 + (people - 250) * 0.0025 + (outdoor - 29) * 0.2
        co2 = int(520 + (people - 250) * 1.15)
        rows.append(dict(time=t.strftime("%H:%M"),
                         ch2_status="RUNNING", ch2_cond_pump_flow_pct=round(flow, 1),
                         ch2_chw_supply_c=round(chw, 1), ahu3_status="RUNNING",
                         ahu3_supply_air_c=round(ahu, 1), hall_temp_c=round(hall, 1),
                         hall_co2_ppm=co2, hall_people=people, outdoor_temp_c=round(outdoor, 1)))
    return pd.DataFrame(rows)

df = simulate()
df.iloc[::4]

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
df.plot(x="time", y="ch2_cond_pump_flow_pct", ax=ax[0], title="CH-2 pump flow (%)", legend=False, color="#EA4335")
df.plot(x="time", y="ch2_chw_supply_c", ax=ax[1], title="Chilled water supply (°C)", legend=False, color="#4285F4")
df.plot(x="time", y="hall_temp_c", ax=ax[2], title="Main Hall temperature (°C)", legend=False, color="#34A853")
ax[2].axhline(26, ls="--", color="grey"); ax[2].text(0.5, 26.1, "SOP limit 26 °C")
plt.tight_layout(); plt.show()

**Quick question:** if you only had a *chiller dashboard* (status = `RUNNING`) and a *hall thermostat*, how long would it take you to connect the two?

## Mission 2 · Connect the world 🕸️
A time-series table can't tell you that **CH-2 cools the Main Hall**. A graph can.
(In production you'd model this with Google's open-source [Digital Buildings Ontology](https://google.github.io/digitalbuildings/) and store it in [Spanner Graph](https://cloud.google.com/spanner/docs/graph/overview).)

In [ ]:
import networkx as nx
G = nx.DiGraph()
edges = [("CH-2", "AHU-3", "FEEDS"), ("CH-1", "AHU-3", "CAN_FEED"),
         ("AHU-3", "Main Hall", "SERVES"), ("DevFest", "Main Hall", "OCCUPIES"),
         ("Main Hall", "Gate 1", "EXITS_VIA"), ("Main Hall", "Gate 3", "EXITS_VIA")]
for a, b, rel in edges:
    G.add_edge(a, b, rel=rel)

def what_affects(space):
    # everything upstream of a space in the graph
    return sorted(nx.ancestors(G, space))

print("Things that can make the Main Hall hot or cold:", what_affects("Main Hall"))
print("Exits the crowd will use:", list(G.successors("Main Hall")))

plt.figure(figsize=(9, 4))
pos = {"CH-2": (0, 1), "CH-1": (0, 0), "AHU-3": (1, 0.5), "DevFest": (2, 1.3), "Main Hall": (2, 0.5), "Gate 1": (3, 1), "Gate 3": (3, 0)}
nx.draw(G, pos, with_labels=True, node_size=2600, node_color="#C3ECF6", edgecolors="#1E1E1E", font_size=9)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "rel"), font_size=7)
plt.show()

### The rules of the world (SOP) and the world outside (weather)
In production the forecast would come from the [Google Maps Platform Weather API](https://developers.google.com/maps/documentation/weather/overview).

In [ ]:
SOP = {
    "chw_supply_c": (6, 8),
    "hall_temp_max_c": 26,
    "hall_co2_max_ppm": 1000,
    "approval_required": ["start_or_stop_chiller"],
    "never_automatic": ["gates", "fire_and_life_safety"],
}
FORECAST = [{"time": "12:00", "temp_c": 32}, {"time": "13:00", "temp_c": 34}, {"time": "15:00", "temp_c": 35, "condition": "sunny"}]

# Plain rules find WHEN things broke...
first = lambda cond: df.loc[cond, "time"].iloc[0] if cond.any() else None
print("Pump flow first dropped:     ", first(df.ch2_cond_pump_flow_pct < 100))
print("Chilled water left the SOP:  ", first(df.ch2_chw_supply_c > SOP["chw_supply_c"][1]))
print("Hall crossed 26 °C:          ", first(df.hall_temp_c > SOP["hall_temp_max_c"]))
print("...but not WHY, or what to do about it. That's Mission 3.")

## Mission 3 · Wake it up 🤖
Now we give Gemini the **map + relationships + rules + forecast** and ask it to reason like a facilities engineer.

1. Get a free API key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey).
2. In Colab, open the 🔑 **Secrets** panel (left sidebar) and add `GEMINI_API_KEY`.
3. Run the cells below.

`MODEL` uses the `gemini-flash-latest` alias. If it errors, pick any current Flash model ID from AI Studio.

In [ ]:
import os, json
MODEL = "gemini-flash-latest"

API_KEY = os.environ.get("GEMINI_API_KEY")
try:
    from google.colab import userdata
    API_KEY = API_KEY or userdata.get("GEMINI_API_KEY")
except Exception:
    pass
print("Gemini key found ✅" if API_KEY else "No key — Mission 3 will use a sample answer (still fun!)")

In [ ]:
context = {
    "telemetry_csv": df[df.index % 3 == 0].to_csv(index=False),
    "graph_edges": [f"{a} {d['rel']} {b}" for a, b, d in G.edges(data=True)],
    "equipment_status": {"CH-2": "RUNNING", "CH-1": "STANDBY", "AHU-3": "RUNNING"},
    "sop": SOP,
    "weather_forecast": FORECAST,
    "event": "DevFest in Main Hall until 17:00, lunch 13:00-14:00",
}

PROMPT = f'''You are a senior campus facilities engineer.
Using the telemetry, the graph, the SOP and the forecast below:
1. Find what's abnormal and which asset it started on.
2. Predict the Main Hall for the next hour.
3. Recommend actions. For each action give a wanted level from 1 to 5 stars:
   1 observe, 2 recommend (human decides), 3 act after human approval, 4 act then report, 5 fully autonomous.

Reply as JSON with keys: finding (string), next_hour (string),
actions (list of objects with: action, asset, category, stars, why).
category must be one of: maintenance, chiller, comfort, gates, fire_and_life_safety, communication.

DATA:
{json.dumps(context, indent=1)}
'''

SAMPLE = {
  "finding": "CH-2's condenser pump flow fell from 100% to about 62% starting around 10:40. Chilled water rose from 7 °C to about 11.6 °C (SOP max 8 °C). CH-2 and AHU-3 still report RUNNING, so status alone hid it.",
  "next_hour": "With about 620 people inside and 34 °C outside by 13:00, the Main Hall is likely to pass 28 °C before lunch.",
  "actions": [
    {"action": "Send a technician to inspect CH-2's condenser pump", "asset": "CH-2", "category": "maintenance", "stars": 2, "why": "Root cause; needs human eyes"},
    {"action": "Start standby chiller CH-1 to serve AHU-3", "asset": "CH-1", "category": "chiller", "stars": 4, "why": "Fastest way to restore cooling"},
    {"action": "Ask organisers to bring the break forward and open side doors", "asset": "Main Hall", "category": "communication", "stars": 2, "why": "Reduce heat load"},
    {"action": "Open Gate 3 for the session exit", "asset": "Gate 3", "category": "gates", "stars": 3, "why": "Avoid a crush at Gate 1"}
  ]
}

if API_KEY:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=API_KEY)
    resp = client.models.generate_content(
        model=MODEL, contents=PROMPT,
        config=types.GenerateContentConfig(response_mime_type="application/json"))
    answer = json.loads(resp.text)
else:
    answer = SAMPLE

print("FINDING:  ", answer["finding"]); print("NEXT HOUR:", answer["next_hour"])
pd.DataFrame(answer["actions"])

### The guardrail: rules + physics decide, not the model ⭐
The model *proposes*. Plain, deterministic code enforces the **wanted-level ladder** from the SOP.
(The sample answer deliberately over-reaches on two actions, so you can watch the guardrail catch them.)

In [ ]:
MAX_STARS = {"gates": 2, "fire_and_life_safety": 1, "chiller": 3}  # chiller changes need approval (3★)

def guardrail(actions):
    out = []
    for a in actions:
        proposed = int(a.get("stars", 1))
        allowed = min(proposed, MAX_STARS.get(a.get("category"), 4))
        out.append({**a, "proposed": "★" * proposed, "allowed": "★" * allowed, "capped": allowed < proposed})
    return pd.DataFrame(out)[["action", "category", "proposed", "allowed", "capped"]]

guardrail(answer["actions"])

## Bonus · Go offline with Gemma 💻
Many campuses and industrial parks can't send building data to the cloud. Run the **same prompt** on an open Gemma model on your laptop with [Ollama](https://ollama.com):

```bash
ollama pull gemma3          # or the newest Gemma model listed on ollama.com
ollama run gemma3 "$(cat prompt.txt)"
```

In [ ]:
open("prompt.txt", "w").write(PROMPT)
print("Saved prompt.txt — download it from the Files panel and try it with Gemma locally.")

## Side quests 🎮
1. **Change the villain:** call `simulate(drift_starts_min=20)` or make the crowd grow faster. Does Gemini still find the root cause?
2. **Add a sensor that lies:** make `hall_temp_c` read 2 °C low. Can the agent spot that the *mirror* is wrong?
3. **Make your campus aware:** swap the hall for your canteen queue, parking lot or library AC. What's the map? The graph? The star level?
4. **Wrap it in an agent:** turn `what_affects()` and `guardrail()` into tools with Google's [Agent Development Kit (ADK)](https://google.github.io/adk-docs/).

Built something? Share it with **#DevFestSiliguri** and tag Kazi on [LinkedIn](https://www.linkedin.com/in/haquekazi/).